In [39]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer

print("Hello World! (ruleaza!!!)")

train_coords = np.load("train_coordinates.npy")
with open("train_samples.txt", "r", encoding="utf-8") as f:
    train_samples = np.array(f.read().splitlines())
with open("test_samples.txt", "r", encoding="utf-8") as f:
    test_samples = np.array(f.read().splitlines())


t_coords = np.array(train_coords)
kf = KFold(n_splits=3, shuffle=True, random_state=42)

def transform_data(train_data, val_data):
    vectorizer = TfidfVectorizer(max_features=5000, sublinear_tf=True)

    train_features = vectorizer.fit_transform(train_data).toarray()
    val_features = vectorizer.transform(val_data).toarray()

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)
    val_scaled = scaler.transform(val_features)
    return train_scaled, val_scaled

# caut bst alpha pentru regresia ridge
def ridge_ex():
    best_a = 0
    best_mean_mse = float('inf')

    for a in [3500, 4000]:
        ridge = Ridge(alpha=a)
        mse_scores = []

        for train_index, val_index in kf.split(train_samples):
            X_train, X_val = train_samples[train_index], train_samples[val_index]
            y_train, y_val = t_coords[train_index], t_coords[val_index]

            X_train_scaled, X_val_scaled = transform_data(X_train, X_val)

            ridge.fit(X_train_scaled, y_train)
            predictions = ridge.predict(X_val_scaled)

            mse_scores.append(mean_squared_error(y_val, predictions))

        mean_mse = np.mean(mse_scores)
        print(f"Alpha = {a} -> MSE mediu: {mean_mse:.4f}")

        if mean_mse < best_mean_mse:
            best_mean_mse = mean_mse
            best_a = a

    print(f"\nCel mai bun alpha selectat: {best_a} cu MSE mediu: {best_mean_mse:.4f}")
    return best_a

# il folosesc pt predictionul final
def ridge_regression_final(alpha):
    vectorizer = TfidfVectorizer(max_features=5000, sublinear_tf=True)
    scaler = StandardScaler()

    X_train_full = scaler.fit_transform(vectorizer.fit_transform(train_samples).toarray())
    X_test_full = scaler.transform(vectorizer.transform(test_samples).toarray())

    final_ridge = Ridge(alpha=alpha)
    final_ridge.fit(X_train_full, t_coords)

    predictions = final_ridge.predict(X_test_full)
    return predictions

best_alpha = ridge_ex()
final_predictions = ridge_regression_final(best_alpha)

# am mse around .79
with open("Robitu_RianaIoana_244_subiect1_solutia1.txt", "w", encoding="utf-8") as f:
    for prediction in final_predictions:
        x, y = prediction[0], prediction[1]
        f.write(f'{x} {y}\n')


Hello World! (ruleaza!!!)
[[51.88486301 10.48784091]
 [51.88486301 10.48784091]
 [50.96534247  9.34954545]
 [52.54691781  9.65022727]
 [50.26650685  9.13477273]
 [50.0090411  10.05829545]
 [52.62047945 10.29454545]
 [52.51013699 10.93886364]
 [52.54691781 10.78852273]
 [51.59061644  7.45954545]]


KeyboardInterrupt: 